[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/17_feature_engineering/02_categorical_encoding.ipynb)

# Bài 02 — Mã hoá feature phân loại

**🎯 Mục tiêu**
- One-hot (kèm `handle_unknown`, gộp category hiếm), ordinal, frequency encoding, hashing.
- **Target encoding**: mạnh với feature nhiều giá trị (high cardinality) — và **bẫy rò rỉ** nếu làm ngây thơ; cross-fitting (`TargetEncoder`) và smoothing.
- Native categorical trong HistGradientBoosting / LightGBM / CatBoost.
- Chọn encoding theo model và cardinality.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/17_feature_engineering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Dữ liệu: cửa hàng với 800 merchant

Dự đoán giao dịch bị khiếu nại. Feature: `merchant_id` (800 giá trị — **high cardinality**, mỗi merchant có tỉ lệ khiếu nại riêng), `channel` (3 giá trị), `size` (S/M/L — **có thứ tự**), `amount`.

In [ ]:
n, n_merchants = 20000, 800
merchant_effect = rng.normal(0, 1.2, n_merchants)
merchant_pop = 1 / np.arange(1, n_merchants + 1) ** 1.1                        # phân phối đuôi dài: vài merchant rất phổ biến, đa số hiếm
merchant = rng.choice(n_merchants, n, p=merchant_pop / merchant_pop.sum())
channel = rng.choice(["app", "web", "pos"], n, p=[0.5, 0.3, 0.2])
size = rng.choice(["S", "M", "L"], n)
amount = rng.lognormal(5, 1, n)
logit = -3.0 + merchant_effect[merchant] + 0.5 * (channel == "web") + 0.3 * pd.Series(size).map({"S": 0, "M": 1, "L": 2}).to_numpy() \
        + 0.2 * (np.log(amount) - 5)
y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
df = pd.DataFrame({"merchant_id": merchant.astype(str), "channel": channel, "size": size, "amount": amount})
print(f"tỉ lệ khiếu nại {y.mean():.3f} | số merchant xuất hiện: {df.merchant_id.nunique()} | "
      f"merchant xuất hiện ≤ 5 lần: {(df.merchant_id.value_counts() <= 5).sum()}")

## 2. One-hot, ordinal

- **One-hot**: mỗi category một cột 0/1. Không giả định thứ tự. Với 800 category → 800 cột (thưa) — ổn với model tuyến tính có regularization, nặng với cây.
- **Ordinal**: map sang số nguyên. Hợp lý khi category **có thứ tự** (S < M < L); với category không thứ tự, model tuyến tính sẽ hiểu sai ("web" = 2 × "app"?), còn cây thì vẫn dùng được.
- `handle_unknown="ignore"`: category mới ở test → vector 0; `min_frequency`: gộp category hiếm thành "infrequent".

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False).fit(df[["merchant_id"]])
print(f"one-hot merchant_id: {len(ohe.get_feature_names_out())} cột (merchant < 20 lần gộp chung vào 'infrequent')")
print(OrdinalEncoder(categories=[["S", "M", "L"]]).fit_transform(df[["size"]].head(5)).ravel(), "←", df["size"].head(5).tolist())

## 3. Target encoding và bẫy rò rỉ

Thay mỗi category bằng **trung bình target** của category đó. Một cột duy nhất mang rất nhiều thông tin — nhưng nếu tính trung bình trên **chính** các mẫu mà model sẽ học, mẫu "thấy" nhãn của chính nó: merchant xuất hiện 1 lần → giá trị encoding = đúng nhãn của mẫu đó → **rò rỉ nhãn**.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=0.3, random_state=0, stratify=y)


def naive_target_encode(train_col, y_train, col):
    means = pd.Series(y_train).groupby(train_col.to_numpy()).mean()
    return col.map(means).fillna(y_train.mean()).to_numpy()


te_tr = naive_target_encode(X_tr.merchant_id, y_tr, X_tr.merchant_id)       # ❌ tính trên chính tập train
te_te = naive_target_encode(X_tr.merchant_id, y_tr, X_te.merchant_id)
m_naive = HistGradientBoostingClassifier(random_state=0).fit(np.c_[te_tr, np.log(X_tr.amount)], y_tr)
p_tr = m_naive.predict_proba(np.c_[te_tr, np.log(X_tr.amount)])[:, 1]
p_te = m_naive.predict_proba(np.c_[te_te, np.log(X_te.amount)])[:, 1]
print(f"target encoding NGÂY THƠ: ROC-AUC train = {roc_auc_score(y_tr, p_tr):.3f} | test = {roc_auc_score(y_te, p_te):.3f}")

Train rất cao, test tụt hẳn: model học "tin" cột encoding vì trên train nó gần như là nhãn (với merchant hiếm) → **ước lượng trên train hoàn toàn sai lệch**, và model dựa quá nhiều vào cột này. **Cách đúng:**
1. **Cross-fitting**: chia train thành $k$ fold; encoding cho mẫu ở fold $i$ được tính từ các fold **khác** → không mẫu nào thấy nhãn của chính nó. `sklearn.preprocessing.TargetEncoder` làm điều này trong `fit_transform`.
2. **Smoothing**: kéo trung bình của category ít mẫu về trung bình chung: $\hat\mu_c = \frac{n_c\,\bar y_c + m\,\bar y}{n_c + m}$.

In [ ]:
def smoothed_means(cat, yv, m=20):
    stats = pd.DataFrame({"c": cat, "y": yv}).groupby("c")["y"].agg(["mean", "count"])
    prior = yv.mean()
    return (stats["count"] * stats["mean"] + m * prior) / (stats["count"] + m)


counts = X_tr.merchant_id.value_counts()
raw_means = pd.Series(y_tr).groupby(X_tr.merchant_id.to_numpy()).mean()
smooth = smoothed_means(X_tr.merchant_id.to_numpy(), y_tr)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(counts[raw_means.index], raw_means, s=10, alpha=0.5, label="trung bình thô")
ax.scatter(counts[smooth.index], smooth, s=10, alpha=0.5, label="smoothing (m = 20)")
ax.axhline(y_tr.mean(), color="k", ls=":", label="tỉ lệ chung")
ax.set_xscale("log")
ax.set_xlabel("số mẫu của merchant (train)")
ax.set_ylabel("encoding")
ax.legend()
ax.set_title("Merchant ít mẫu: trung bình thô cực đoan (0 hoặc 1); smoothing kéo về tỉ lệ chung")
plt.show()

In [ ]:
enc = TargetEncoder(target_type="binary", random_state=0)
te_tr_cf = enc.fit_transform(X_tr[["merchant_id"]], y_tr)                # cross-fitting bên trong fit_transform
te_te_cf = enc.transform(X_te[["merchant_id"]])
m_cf = HistGradientBoostingClassifier(random_state=0).fit(np.c_[te_tr_cf, np.log(X_tr.amount)], y_tr)
print(f"TargetEncoder (cross-fit): ROC-AUC train = {roc_auc_score(y_tr, m_cf.predict_proba(np.c_[te_tr_cf, np.log(X_tr.amount)])[:, 1]):.3f} | "
      f"test = {roc_auc_score(y_te, m_cf.predict_proba(np.c_[te_te_cf, np.log(X_te.amount)])[:, 1]):.3f}")

Với cross-fitting, khoảng cách train–test **thu hẹp rõ rệt** (phần còn lại là overfit thông thường của model cây), và test không thua cách ngây thơ — encoding trên train "trông giống" encoding mà model sẽ gặp trên dữ liệu mới, nên điểm số trên train/validation lại có ý nghĩa.

## 4. So sánh các cách mã hoá (CV đúng quy trình)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)
num_pipe = make_pipeline(StandardScaler())
log_amount = df.assign(amount=np.log(df.amount))
encoders = {
    "bỏ merchant_id": ColumnTransformer([("cat", OneHotEncoder(), ["channel"]), ("ord", OrdinalEncoder(categories=[["S", "M", "L"]]), ["size"]),
                                         ("num", num_pipe, ["amount"])], sparse_threshold=0),
    "one-hot merchant (min_freq=5)": ColumnTransformer([("m", OneHotEncoder(handle_unknown="ignore", min_frequency=5), ["merchant_id"]),
                                                        ("cat", OneHotEncoder(), ["channel"]),
                                                        ("ord", OrdinalEncoder(categories=[["S", "M", "L"]]), ["size"]),
                                                        ("num", num_pipe, ["amount"])], sparse_threshold=0),
    "TargetEncoder (cross-fit)": ColumnTransformer([("m", TargetEncoder(target_type="binary", random_state=0), ["merchant_id"]),
                                                    ("cat", OneHotEncoder(), ["channel"]),
                                                    ("ord", OrdinalEncoder(categories=[["S", "M", "L"]]), ["size"]),
                                                    ("num", num_pipe, ["amount"])], sparse_threshold=0),
}
rows = []
for enc_name, enc in encoders.items():
    for model_name, model in [("LogReg", LogisticRegression(C=1.0, max_iter=3000)), ("HistGB", HistGradientBoostingClassifier(random_state=0))]:
        auc = cross_val_score(make_pipeline(enc, model), log_amount, y, cv=cv, scoring="roc_auc").mean()
        rows.append({"encoding": enc_name, "model": model_name, "ROC-AUC (CV)": auc})
pd.DataFrame(rows).pivot(index="encoding", columns="model", values="ROC-AUC (CV)").round(4)

- Bỏ `merchant_id` mất phần lớn tín hiệu.
- One-hot + model tuyến tính có regularization dùng được high cardinality (mỗi merchant một hệ số, merchant hiếm bị co về 0).
- TargetEncoder (cross-fit) nén 800 category vào **1 cột** giàu thông tin — rất hợp với cây.

## 5. Native categorical và các encoding khác

In [ ]:
Xn = log_amount.copy()
top = Xn.merchant_id.value_counts().index[:250]                        # sklearn: tối đa 255 category / feature
Xn["merchant_id"] = Xn.merchant_id.where(Xn.merchant_id.isin(top), "other")
for c in ["merchant_id", "channel", "size"]:
    Xn[c] = Xn[c].astype("category")
native = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=0)
print(f"HistGB native categorical: ROC-AUC (CV) = {cross_val_score(native, Xn, y, cv=cv, scoring='roc_auc').mean():.4f}")

HistGradientBoosting xử lý trực tiếp category bằng cách sắp xếp category theo gradient ở mỗi split. Giới hạn: sklearn chỉ nhận tối đa 255 category mỗi feature → ta gộp merchant hiếm vào `"other"` (LightGBM/CatBoost không giới hạn như vậy; CatBoost còn tự dùng ordered target statistics).

| Encoding | Ý tưởng | Hợp với |
|---|---|---|
| One-hot | 1 cột / category | tuyến tính, ít category |
| Ordinal | số nguyên theo thứ tự | category có thứ tự; cây |
| Frequency / count | số lần xuất hiện | cây; khi tần suất có ý nghĩa |
| **Target (cross-fit + smoothing)** | trung bình target | high cardinality, cây & tuyến tính |
| Hashing | băm vào $k$ cột cố định | cực nhiều category, streaming |
| Embedding | vector học được | deep learning (chương 26, 32) |
| Native | model tự xử lý | HistGB, LightGBM, CatBoost |

## 🧠 Kiểm tra nhanh

1. Vì sao target encoding "ngây thơ" làm ROC-AUC train cao bất thường?
2. Smoothing với $m = 20$: merchant có 2 mẫu, cả 2 đều bị khiếu nại; tỉ lệ chung 0.15. Encoding bằng bao nhiêu?
3. Khi nào ordinal encoding hợp lý cho model tuyến tính?
4. Category mới xuất hiện ở test — mỗi encoding xử lý thế nào?

## 📌 Tóm tắt

- One-hot cho ít category; ordinal cho category có thứ tự; target encoding cho high cardinality — **luôn cross-fit + smoothing**.
- Mọi encoding học từ dữ liệu phải nằm trong Pipeline.
- Cây hiện đại (HistGB, LightGBM, CatBoost) xử lý category native.

## 📚 Tài liệu tham khảo

- Micci-Barreca (2001) — *A preprocessing scheme for high-cardinality categorical attributes*
- Pargent et al. (2022) — *Regularized target encoding outperforms traditional methods in supervised machine learning with high cardinality features*
- scikit-learn User Guide — Target Encoder